# Cs_copilot Robustness Testing

This notebook demonstrates how to use the robustness testing framework to assess how consistent the Cs_copilot pipeline is when given semantically equivalent but syntactically different prompts.

## What is Robustness Testing?

Robustness testing checks whether the system produces consistent outputs when the same task is requested in different ways. For example:
- "Fetch kinase 2 inhibitor data" vs "Download CHEMBL data for kinase 2 inhibitors"
- "Build GTM" vs "Construct the manifold" vs "Create a GTM model"

The framework compares outputs across multiple runs and calculates a robustness score based on:
- **Data consistency** (40%): Are datasets identical?
- **Semantic similarity** (30%): Are text responses semantically equivalent?
- **Process consistency** (20%): Do runs complete successfully?
- **Visual similarity** (10%): Are visualizations consistent?


## Setup


In [63]:
%load_ext autoreload
%autoreload 2
from dotenv import load_dotenv
load_dotenv()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


True

In [64]:
import sys
import os
import logging
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import shutil
from datetime import datetime
import uuid
import json
# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


In [65]:
# Import Cs_copilot
from cs_copilot.agents import get_cs_copilot_agent_team
from cs_copilot.storage import S3


In [66]:
# Import robustness testing framework
sys.path.insert(0, str(Path.cwd().parent / 'tests' / 'robustness'))
from prompt_variations import PromptVariationGenerator
from comparators import OutputComparator
from metrics import RobustnessMetrics


In [67]:
# Setup API key
from getpass import getpass
deepseek_api_key = getpass("DeepSeek API key: ")


DeepSeek API key:  ········


In [68]:
from agno.models.deepseek import DeepSeek
model = DeepSeek(id="deepseek-chat", api_key=deepseek_api_key)


## 1. Explore Available Prompt Variations


In [69]:
# Initialize the prompt variation generator
generator = PromptVariationGenerator()

# List all available prompt categories
available_prompts = generator.list_available_prompts()
print("Available prompt categories:")
for prompt_key in available_prompts:
    print(f"  - {prompt_key}")


2025-12-29 15:56:57,319 - prompt_variations - INFO - Loaded prompt templates from /data/aorlov/cs_copilot/tests/robustness/fixtures/prompt_templates.yaml
2025-12-29 15:56:57,320 - prompt_variations - INFO - Loading sentence transformer model for similarity validation
2025-12-29 15:56:57,324 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cuda:0
2025-12-29 15:56:57,325 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Available prompt categories:
  - full_pipeline
  - chembl_download
  - gtm_optimization
  - density_analysis
  - activity_analysis
  - chemotype_analysis
  - autoencoder_sampling
  - gtm_guided_sampling
  - interpolation
  - latent_exploration


In [70]:
# Get variations for a specific category
prompt_key = "chembl_download"  # Change this to try different categories
variations = generator.get_variations(prompt_key, n=5)

print(f"\n{len(variations)} variations for '{prompt_key}':\n")
for i, variation in enumerate(variations, 1):
    print(f"{i}. {variation}")
    print()


2025-12-29 15:56:59,600 - prompt_variations - INFO - Validating 4 variations for 'chembl_download'


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-29 15:56:59,635 - prompt_variations - WARNING - Variation 1 for 'chembl_download' has low similarity to base


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


5 variations for 'chembl_download':

1. Download kinase 2 inhibitor data from ChEMBL

2. Fetch CDK2 bioactivity data from ChEMBL database

3. Retrieve cyclin-dependent kinase 2 inhibitor compounds from ChEMBL

4. Get kinase 2 inhibitor bioactivity information from ChEMBL

5. Pull down ChEMBL data for CDK2 inhibitors



## 2. Validate Semantic Similarity of Variations

The framework uses sentence transformers to ensure variations maintain semantic similarity to the base prompt.


In [71]:
# Check semantic similarity
base = variations[0]
print(f"Base prompt: {base}\n")

for i, var in enumerate(variations[1:], 2):
    is_valid = generator.validate_variation(base, var, min_similarity=0.50)
    print(f"Variation {i}: {'✅ Valid' if is_valid else '❌ Invalid'}")
    print(f"  {var}\n")


Base prompt: Download kinase 2 inhibitor data from ChEMBL



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Variation 2: ✅ Valid
  Fetch CDK2 bioactivity data from ChEMBL database



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Variation 3: ✅ Valid
  Retrieve cyclin-dependent kinase 2 inhibitor compounds from ChEMBL



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Variation 4: ✅ Valid
  Get kinase 2 inhibitor bioactivity information from ChEMBL



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Variation 5: ✅ Valid
  Pull down ChEMBL data for CDK2 inhibitors



## 3. Working with the Prompt Templates YAML File

The prompt variations are stored in `tests/robustness/fixtures/prompt_templates.yaml`. Let's explore and validate them.


In [72]:
# View the YAML file structure
yaml_path = Path.cwd().parent / 'tests' / 'robustness' / 'fixtures' / 'prompt_templates.yaml'
with open(yaml_path, 'r') as f:
    templates = yaml.safe_load(f)

print("Available prompt categories in YAML:")
print("="*60)
for category in templates['prompts'].keys():
    base = templates['prompts'][category]['base']
    num_variations = len(templates['prompts'][category]['variations'])
    print(f"\n{category}")
    print(f"  Base: {base[:80]}...")
    print(f"  Variations: {num_variations}")


Available prompt categories in YAML:

full_pipeline
  Base: Fetch kinase 2 inhibitor data, prepare GTM, describe the density landscape, prep...
  Variations: 10

chembl_download
  Base: Download kinase 2 inhibitor data from ChEMBL...
  Variations: 10

gtm_optimization
  Base: Build a GTM map...
  Variations: 10

density_analysis
  Base: Describe the density landscape...
  Variations: 10

activity_analysis
  Base: Create and describe the activity landscape...
  Variations: 10

chemotype_analysis
  Base: Find dense nodes and analyze chemotypes. What assays? Are actives separated?...
  Variations: 10

autoencoder_sampling
  Base: Sample from autoencoder model...
  Variations: 10

gtm_guided_sampling
  Base: Sample from autoencoder model using GTM dense nodes as guidance...
  Variations: 10

interpolation
  Base: Interpolate between two molecules using the autoencoder...
  Variations: 10

latent_exploration
  Base: Explore the latent space around a given molecule using the autoencoder...
 

In [ ]:
# Compare variation styles for a specific category
category = "gtm_optimization"  # Try: chembl_download, density_analysis, etc.

print(f"Variations for '{category}':")
print("="*60)

base = templates['prompts'][category]['base']
variations_yaml = templates['prompts'][category]['variations']

print(f"\n📌 BASE: {base}\n")

for i, var in enumerate(variations_yaml, 1):
    # Calculate length to show variation style
    length_indicator = "Short" if len(var) < 50 else "Medium" if len(var) < 100 else "Long"
    
    print(f"{i}. [{length_indicator:6s}] {var}")


In [ ]:
# Validate all variations in a category using semantic similarity
from sentence_transformers import SentenceTransformer, util

category = "gtm_optimization"
base = templates['prompts'][category]['base']
variations_yaml = templates['prompts'][category]['variations']

print(f"Semantic Similarity Validation for '{category}':")
print("="*60)

model_bert = SentenceTransformer("all-MiniLM-L6-v2")

# Encode base
base_embedding = model_bert.encode(base)

print(f"\nBase: {base}\n")

for i, var in enumerate(variations_yaml, 1):
    # Calculate similarity
    var_embedding = model_bert.encode(var)
    similarity = util.cos_sim(base_embedding, var_embedding).item()
    
    # Visual indicator
    status = "✅" if similarity >= 0.70 else "⚠️" if similarity >= 0.60 else "❌"
    bar = "█" * int(similarity * 30)
    
    print(f"{i:2d}. {status} {similarity:.3f} {bar}")
    print(f"    {var[:80]}...")
    print()


In [ ]:
# Test a new variation programmatically (not saved to file)
new_variation = "Please construct a Generative Topographic Map for the dataset"

print("Testing a new variation:")
print(f"New: {new_variation}\n")

# Check similarity with existing base
base = templates['prompts']['gtm_optimization']['base']
new_embedding = model_bert.encode(new_variation)
base_embedding = model_bert.encode(base)
similarity = util.cos_sim(base_embedding, new_embedding).item()

print(f"Base: {base}")
print(f"\nSimilarity: {similarity:.3f}")
print(f"Valid: {'✅ Yes' if similarity >= 0.70 else '❌ No'} (threshold: 0.70)")

# Test with the generator too
is_valid = generator.validate_variation(base, new_variation, min_similarity=0.70)
print(f"Generator validation: {is_valid}")


In [ ]:
# Analyze variation diversity across all categories
print("Variation Statistics Across All Categories:")
print("="*60)

stats = []
for category, data in templates['prompts'].items():
    base = data['base']
    variations_data = data['variations']
    
    # Calculate stats
    avg_length = sum(len(v) for v in variations_data) / len(variations_data)
    min_length = min(len(v) for v in variations_data)
    max_length = max(len(v) for v in variations_data)
    
    stats.append({
        'category': category,
        'num_variations': len(variations_data),
        'avg_length': avg_length,
        'min_length': min_length,
        'max_length': max_length,
        'base_length': len(base)
    })

# Display as DataFrame
df_stats = pd.DataFrame(stats)
print(df_stats.to_string(index=False))

print(f"\nTotal prompt categories: {len(templates['prompts'])}")
print(f"Total variations: {sum(s['num_variations'] for s in stats)}")


In [ ]:
# Visualize semantic similarity heatmap for one category
import matplotlib.pyplot as plt

category = "chembl_download"
base = templates['prompts'][category]['base']
variations_viz = templates['prompts'][category]['variations']

all_prompts = [base] + variations_viz
embeddings = model_bert.encode(all_prompts)

# Calculate similarity matrix
n = len(all_prompts)
similarity_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        similarity_matrix[i, j] = util.cos_sim(embeddings[i], embeddings[j]).item()

# Plot heatmap
plt.figure(figsize=(10, 8))
plt.imshow(similarity_matrix, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(label='Cosine Similarity')
plt.title(f'Semantic Similarity Heatmap: {category}')
plt.xlabel('Prompt Index')
plt.ylabel('Prompt Index')

# Add labels
labels = ['Base'] + [f'V{i}' for i in range(1, len(variations_viz) + 1)]
plt.xticks(range(n), labels, rotation=45)
plt.yticks(range(n), labels)

# Add text annotations
for i in range(n):
    for j in range(n):
        text = plt.text(j, i, f'{similarity_matrix[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nAverage similarity to base: {similarity_matrix[0, 1:].mean():.3f}")
print(f"Min similarity to base: {similarity_matrix[0, 1:].min():.3f}")
print(f"Max similarity to base: {similarity_matrix[0, 1:].max():.3f}")


## 4. Example: Test Single Step Robustness

Let's test robustness of a single step (ChEMBL data download) with multiple prompt variations.


In [73]:
# Get variations for ChEMBL download
chembl_variations = generator.get_variations("chembl_download", n=3)

# Initialize comparator and metrics
comparator = OutputComparator()
metrics_calc = RobustnessMetrics()

# Setup isolated output directories for this test run
test_run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
base_output_dir = Path("robustness_outputs") / test_run_id / "chembl_download"
base_output_dir.mkdir(parents=True, exist_ok=True)

# Store outputs from each run
outputs = []

# Check if S3/MinIO is enabled
from cs_copilot.storage import is_s3_enabled, get_s3_config

s3_enabled = is_s3_enabled()
if s3_enabled:
    s3_config = get_s3_config()
    print(f"✓ MinIO/S3 storage enabled")
    print(f"  Bucket: {s3_config.bucket_name}")
    print(f"  Endpoint: {s3_config.endpoint_url}")
else:
    print("⚠ MinIO/S3 storage not enabled - using local storage only")

print(f"\nTesting ChEMBL download robustness with {len(chembl_variations)} variations...")
print(f"Local output directory: {base_output_dir}\n")

2025-12-29 15:57:11,225 - prompt_variations - INFO - Validating 2 variations for 'chembl_download'


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-29 15:57:11,250 - prompt_variations - WARNING - Variation 1 for 'chembl_download' has low similarity to base


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-29 15:57:11,278 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cuda:0
2025-12-29 15:57:11,279 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2025-12-29 15:57:13,390 - comparators - INFO - Initialized OutputComparator


✓ MinIO/S3 storage enabled
  Bucket: chatbot-assets
  Endpoint: http://localhost:9000

Testing ChEMBL download robustness with 3 variations...
Local output directory: robustness_outputs/20251229_155713/chembl_download



In [74]:
# Run each variation with isolated session and MinIO/S3 storage
for i, prompt in enumerate(chembl_variations):
    print(f"\n{'='*60}")
    print(f"Run {i+1}/{len(chembl_variations)}")
    print(f"Prompt: {prompt}")
    print(f"{'='*60}\n")
    
    # Create unique session ID for this run
    run_session_id = f"robustness_chembl_{test_run_id}_run{i+1}_{uuid.uuid4().hex[:8]}"
    run_output_dir = base_output_dir / f"run_{i+1}"
    run_output_dir.mkdir(exist_ok=True)
    
    print(f"Session ID: {run_session_id}")
    print(f"Local output directory: {run_output_dir}")
    
    if s3_enabled:
        s3_prefix = f"sessions/{run_session_id}"
        print(f"MinIO/S3 prefix: s3://{s3_config.bucket_name}/{s3_prefix}/")
    print()
    
    # ========================================
    # KEY FIX: Directly modify S3.prefix instead of reloading modules
    # ========================================
    # Reloading modules doesn't work because:
    # 1. ChemblToolkit and other tools import S3 at module load time
    # 2. Reloading creates a NEW S3 class, but tools still reference the OLD one
    # 3. Files get saved with the original prefix, not the reloaded one
    #
    # Solution: Directly modify S3.prefix class attribute, which affects ALL code
    from cs_copilot.storage.client import S3 as S3Client
    original_prefix = S3Client.prefix
    new_prefix = f"sessions/{run_session_id}"
    S3Client.prefix = new_prefix
    
    try:
        # S3.prefix is now set to our unique session prefix
        # All file operations will use this prefix
        from cs_copilot.storage import S3
        
        # Create fresh model instance
        model = DeepSeek(id="deepseek-chat", api_key=deepseek_api_key)
        
        # Create fresh agent with unique session
        # Note: session_id is set via SESSION_ID env var and picked up by reloading the storage module
        agent = get_cs_copilot_agent_team(
            model=model,
            debug_mode=False,
            stream_intermediate_steps=False,
            show_members_responses=False,
        )
        
        # Run the agent
        result = agent.run(prompt, stream=False)
        session_state = agent.get_session_state()
        
        # Collect all generated files (both S3 URLs and file paths)
        generated_files = {}
        s3_files = {}  # Track S3 URLs separately
        
        def download_and_track_file(key, value, prefix=""):
            """Download file from S3 if needed and track both S3 and local paths.
            
            The ChEMBL toolkit saves files with relative paths like "chembl_kinase_2.csv"
            which get stored at s3://bucket/sessions/{session_id}/chembl_kinase_2.csv
            
            Session state stores the RELATIVE filename, not the full S3 URL.
            """
            if not isinstance(value, str) or not value:
                return False
            
            # Skip URLs that are not files
            if value.startswith("http://") or value.startswith("https://"):
                return False
            
            full_key = f"{prefix}.{key}" if prefix else key
            
            # Case 1: Full S3 URL (s3://bucket/path/file.csv)
            if value.startswith("s3://"):
                try:
                    filename = value.split("/")[-1]
                    local_path = run_output_dir / filename
                    
                    print(f"  ⬇ Downloading from S3 URL: {value}")
                    with S3.open(value, "rb") as s3_file:
                        local_path.write_bytes(s3_file.read())
                    
                    s3_files[full_key] = value
                    generated_files[full_key] = str(local_path)
                    print(f"  ✓ Saved locally: {filename}")
                    return True
                    
                except Exception as e:
                    print(f"  ✗ Failed to download S3 URL {value}: {e}")
                    return False
            
            # Case 2: Relative path (most common - e.g., "chembl_kinase_2.csv")
            # This is what ChemblToolkit saves to session state
            if not value.startswith("/"):
                if s3_enabled:
                    try:
                        # Construct full S3 URL using current prefix
                        s3_url = S3.path(value)
                        local_path = run_output_dir / value
                        
                        print(f"  ⬇ Downloading relative path '{value}' from: {s3_url}")
                        with S3.open(value, "rb") as s3_file:
                            local_path.write_bytes(s3_file.read())
                        
                        s3_files[full_key] = s3_url
                        generated_files[full_key] = str(local_path)
                        print(f"  ✓ Saved locally: {value}")
                        return True
                        
                    except Exception as e:
                        print(f"  ✗ Failed to download relative path '{value}': {e}")
                        # Fall through to try as local file
                
                # Try as local file path
                source_path = Path(value)
                if source_path.exists() and source_path.is_file():
                    dest_path = run_output_dir / source_path.name
                    shutil.copy2(source_path, dest_path)
                    generated_files[full_key] = str(dest_path)
                    print(f"  ✓ Copied local file: {source_path.name}")
                    return True
                    
                return False
            
            # Case 3: Absolute local path
            source_path = Path(value)
            if source_path.exists() and source_path.is_file():
                dest_path = run_output_dir / source_path.name
                shutil.copy2(source_path, dest_path)
                generated_files[full_key] = str(dest_path)
                print(f"  ✓ Copied local file: {source_path.name}")
                return True
            
            return False
        
        # Debug: Print session state to see what we're working with
        print(f"\n📊 Session state keys: {list(session_state.keys())}")
        for key, value in session_state.items():
            if isinstance(value, dict):
                print(f"   {key}: {value}")
            elif isinstance(value, str) and len(value) < 200:
                print(f"   {key}: {value}")
            else:
                print(f"   {key}: <{type(value).__name__}>")
        
        # Iterate through session state to find and download files
        print("\n📁 Processing session state for files:")
        for key, value in session_state.items():
            # Handle direct file paths/URLs
            if download_and_track_file(key, value):
                continue
            
            # Handle nested dictionaries (like data_file_paths, gtm_file_paths)
            if isinstance(value, dict):
                for subkey, subvalue in value.items():
                    download_and_track_file(subkey, subvalue, prefix=key)
            
            # Handle lists of file paths/URLs
            elif isinstance(value, list):
                for idx, item in enumerate(value):
                    download_and_track_file(f"{key}[{idx}]", item)
        
        # Store output with complete metadata
        output = {
            "run_id": i,
            "prompt": prompt,
            "session_id": run_session_id,
            "result": result.content,
            "session_state": session_state,
            "output_directory": str(run_output_dir),
            "generated_files": generated_files,  # Local paths
            "s3_files": s3_files,  # S3 URLs
            "s3_prefix": f"s3://{s3_config.bucket_name}/{s3_prefix}/" if s3_enabled else None,
            "timestamp": datetime.now().isoformat()
        }
        outputs.append(output)
        
        print(f"\n✅ Run {i+1} completed successfully")
        print(f"   Local files: {len(generated_files)}")
        if s3_files:
            print(f"   S3 files: {len(s3_files)}")
        print(f"   Files saved to: {run_output_dir}")
        
    except KeyboardInterrupt:
        print(f"\n⚠️  Run {i+1} interrupted by user")
        output = {
            "run_id": i,
            "prompt": prompt,
            "session_id": run_session_id,
            "status": "interrupted",
            "output_directory": str(run_output_dir),
            "s3_prefix": f"s3://{s3_config.bucket_name}/{s3_prefix}/" if s3_enabled else None,
            "timestamp": datetime.now().isoformat()
        }
        outputs.append(output)
        break
        
    except Exception as e:
        print(f"\n❌ Run {i+1} failed: {e}")
        logger.exception(f"Run {i+1} failed with exception:")
        
        # Still save whatever we can
        output = {
            "run_id": i,
            "prompt": prompt,
            "session_id": run_session_id,
            "error": str(e),
            "output_directory": str(run_output_dir),
            "s3_prefix": f"s3://{s3_config.bucket_name}/{s3_prefix}/" if s3_enabled else None,
            "timestamp": datetime.now().isoformat()
        }
        outputs.append(output)
        continue
    
    finally:
        # Restore original S3 prefix
        S3Client.prefix = original_prefix

print(f"\n{'='*60}")
print(f"All runs completed!")
print(f"Local results: {base_output_dir}")
if s3_enabled:
    print(f"S3 results: s3://{s3_config.bucket_name}/sessions/")
print(f"Successful runs: {sum(1 for o in outputs if 'error' not in o and o.get('status') != 'interrupted')}/{len(chembl_variations)}")

# Save run summary
summary = {
    "test_run_id": test_run_id,
    "test_type": "chembl_download",
    "total_runs": len(chembl_variations),
    "successful_runs": sum(1 for o in outputs if 'error' not in o and o.get('status') != 'interrupted'),
    "base_output_directory": str(base_output_dir),
    "s3_enabled": s3_enabled,
    "s3_bucket": s3_config.bucket_name if s3_enabled else None,
    "timestamp": datetime.now().isoformat(),
    "runs": [
        {
            "run_id": o['run_id'],
            "prompt_preview": o['prompt'][:80] + "..." if len(o['prompt']) > 80 else o['prompt'],
            "status": "failed" if 'error' in o else ("interrupted" if o.get('status') == 'interrupted' else "success"),
            "output_directory": o['output_directory'],
            "s3_prefix": o.get('s3_prefix'),
            "local_files_count": len(o.get('generated_files', {})),
            "s3_files_count": len(o.get('s3_files', {})),
            "error": o.get('error', None)
        }
        for o in outputs
    ]
}

summary_path = base_output_dir / "test_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nSummary saved to: {summary_path}")



2025-12-29 15:57:31,539 - cs_copilot.agents.teams - INFO - Creating Cs_copilot Agent Team
2025-12-29 15:57:31,541 - cs_copilot.agents.teams - INFO - Creating ChEMBL Downloader agent
2025-12-29 15:57:31,542 - cs_copilot.agents.factories - INFO - Creating agent: chembl_agent
2025-12-29 15:57:31,543 - cs_copilot.agents.factories - INFO - Successfully created agent: chembl_agent
2025-12-29 15:57:31,544 - cs_copilot.agents.teams - INFO - Successfully created ChEMBL Downloader agent
2025-12-29 15:57:31,545 - cs_copilot.agents.teams - INFO - Creating GTM Optimization agent
2025-12-29 15:57:31,546 - cs_copilot.agents.factories - INFO - Creating agent: gtm_opt_agent
2025-12-29 15:57:31,547 - cs_copilot.agents.factories - INFO - Successfully created agent: gtm_opt_agent
2025-12-29 15:57:31,548 - cs_copilot.agents.teams - INFO - Successfully created GTM Optimization agent
2025-12-29 15:57:31,548 - cs_copilot.agents.teams - INFO - Creating GTM Density Analysis agent
2025-12-29 15:57:31,550 - cs_co


Run 1/3
Prompt: Download kinase 2 inhibitor data from ChEMBL

Session ID: robustness_chembl_20251229_155713_run1_d956c79c
Local output directory: robustness_outputs/20251229_155713/chembl_download/run_1
MinIO/S3 prefix: s3://chatbot-assets/sessions/robustness_chembl_20251229_155713_run1_d956c79c/



2025-12-29 15:57:32,347 - httpx - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-12-29 15:57:40,952 - httpx - INFO - HTTP Request: POST https://os-api.agno.com/telemetry/runs "HTTP/1.1 201 Created"
2025-12-29 15:57:40,979 - cs_copilot.agents.teams - INFO - Creating Cs_copilot Agent Team
2025-12-29 15:57:40,982 - cs_copilot.agents.teams - INFO - Creating ChEMBL Downloader agent
2025-12-29 15:57:40,983 - cs_copilot.agents.factories - INFO - Creating agent: chembl_agent
2025-12-29 15:57:40,984 - cs_copilot.agents.factories - INFO - Successfully created agent: chembl_agent
2025-12-29 15:57:40,985 - cs_copilot.agents.teams - INFO - Successfully created ChEMBL Downloader agent
2025-12-29 15:57:40,986 - cs_copilot.agents.teams - INFO - Creating GTM Optimization agent
2025-12-29 15:57:40,988 - cs_copilot.agents.factories - INFO - Creating agent: gtm_opt_agent
2025-12-29 15:57:40,989 - cs_copilot.agents.factories - INFO - Successfully created agent: g


📊 Session state keys: []

📁 Processing session state for files:

✅ Run 1 completed successfully
   Local files: 0
   Files saved to: robustness_outputs/20251229_155713/chembl_download/run_1

Run 2/3
Prompt: Fetch CDK2 bioactivity data from ChEMBL database

Session ID: robustness_chembl_20251229_155713_run2_a5d53e7b
Local output directory: robustness_outputs/20251229_155713/chembl_download/run_2
MinIO/S3 prefix: s3://chatbot-assets/sessions/robustness_chembl_20251229_155713_run2_a5d53e7b/



2025-12-29 15:57:41,170 - cs_copilot.tools.chemistry.autoencoder_toolkit - INFO - Autoencoder model loaded successfully from /home/aorlov/.cache/cs_copilot/models/autoencoder
2025-12-29 15:57:41,171 - cs_copilot.tools.chemistry.autoencoder_toolkit - INFO -   Vocabulary size: 30
2025-12-29 15:57:41,172 - cs_copilot.tools.chemistry.autoencoder_toolkit - INFO -   Latent dimension: 256
2025-12-29 15:57:41,172 - cs_copilot.tools.chemistry.autoencoder_toolkit - INFO -   Device: cuda
2025-12-29 15:57:41,174 - cs_copilot.agents.factories - INFO - Creating agent: autoencoder_gtm_sampling_agent
2025-12-29 15:57:41,175 - cs_copilot.agents.factories - INFO - Successfully created agent: autoencoder_gtm_sampling_agent
2025-12-29 15:57:41,176 - cs_copilot.agents.teams - INFO - Successfully created Autoencoder GTM Sampling agent
2025-12-29 15:57:41,176 - cs_copilot.agents.teams - INFO - Successfully created Cs_copilot Agent Team
2025-12-29 15:57:41,564 - httpx - INFO - HTTP Request: POST https://api.d

INFO Team run c5904269-649a-4a15-987e-870d139b1378 was cancelled

2025-12-29 15:58:34,380 - cs_copilot.agents.teams - INFO - Creating Cs_copilot Agent Team
2025-12-29 15:58:34,383 - cs_copilot.agents.teams - INFO - Creating ChEMBL Downloader agent
2025-12-29 15:58:34,385 - cs_copilot.agents.factories - INFO - Creating agent: chembl_agent
2025-12-29 15:58:34,386 - cs_copilot.agents.factories - INFO - Successfully created agent: chembl_agent
2025-12-29 15:58:34,387 - cs_copilot.agents.teams - INFO - Successfully created ChEMBL Downloader agent
2025-12-29 15:58:34,388 - cs_copilot.agents.teams - INFO - Creating GTM Optimization agent
2025-12-29 15:58:34,391 - cs_copilot.agents.factories - INFO - Creating agent: gtm_opt_agent
2025-12-29 15:58:34,392 - cs_copilot.agents.factories - INFO - Successfully created agent: gtm_opt_agent
2025-12-29 15:58:34,394 - cs_copilot.agents.teams - INFO - Successfully created GTM Optimization agent
2025-12-29 15:58:34,395 - cs_copilot.agents.teams - INFO - Creating GTM Density Analysis agent
2025-12-29 15:58:34,396 - cs_co


📊 Session state keys: []

📁 Processing session state for files:

✅ Run 2 completed successfully
   Local files: 0
   Files saved to: robustness_outputs/20251229_155713/chembl_download/run_2

Run 3/3
Prompt: Retrieve cyclin-dependent kinase 2 inhibitor compounds from ChEMBL

Session ID: robustness_chembl_20251229_155713_run3_f6f34bed
Local output directory: robustness_outputs/20251229_155713/chembl_download/run_3
MinIO/S3 prefix: s3://chatbot-assets/sessions/robustness_chembl_20251229_155713_run3_f6f34bed/



2025-12-29 15:58:34,575 - cs_copilot.tools.chemistry.autoencoder_toolkit - INFO - Autoencoder model loaded successfully from /home/aorlov/.cache/cs_copilot/models/autoencoder
2025-12-29 15:58:34,576 - cs_copilot.tools.chemistry.autoencoder_toolkit - INFO -   Vocabulary size: 30
2025-12-29 15:58:34,576 - cs_copilot.tools.chemistry.autoencoder_toolkit - INFO -   Latent dimension: 256
2025-12-29 15:58:34,577 - cs_copilot.tools.chemistry.autoencoder_toolkit - INFO -   Device: cuda
2025-12-29 15:58:34,580 - cs_copilot.agents.factories - INFO - Creating agent: autoencoder_gtm_sampling_agent
2025-12-29 15:58:34,581 - cs_copilot.agents.factories - INFO - Successfully created agent: autoencoder_gtm_sampling_agent
2025-12-29 15:58:34,582 - cs_copilot.agents.teams - INFO - Successfully created Autoencoder GTM Sampling agent
2025-12-29 15:58:34,583 - cs_copilot.agents.teams - INFO - Successfully created Cs_copilot Agent Team
2025-12-29 15:58:35,240 - __main__ - ERROR - Run 3 failed with exception:


❌ Run 3 failed: Session not found

All runs completed!
Local results: robustness_outputs/20251229_155713/chembl_download
S3 results: s3://chatbot-assets/sessions/
Successful runs: 2/3

Summary saved to: robustness_outputs/20251229_155713/chembl_download/test_summary.json


In [21]:
# Compare text outputs
if len(outputs) >= 2:
    texts = [out["result"] for out in outputs]
    text_comparison = comparator.compare_text_outputs(texts)
    
    print("\nText Comparison Results:")
    print("="*60)
    for metric, value in text_comparison.items():
        if isinstance(value, (int, float)):
            print(f"  {metric:.<40} {value:.3f}")
        else:
            print(f"  {metric:.<40} {value}")
else:
    print("Not enough successful runs to compare")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-29 14:34:46,411 - comparators - INFO - Text comparison: {'semantic_similarity': np.float64(0.7120450735092163), 'entity_overlap': np.float64(0.11143063285920429), 'numeric_consistency': np.float64(0.3283131008443897), 'structural_match': np.float64(0.14056776556776557)}



Text Comparison Results:
  semantic_similarity..................... 0.712
  entity_overlap.......................... 0.111
  numeric_consistency..................... 0.328
  structural_match........................ 0.141


In [22]:
# Compare datasets if available
# Compare datasets from all runs using isolated files
datasets = []
for output in outputs:
    if 'error' in output or output.get('status') == 'interrupted':
        continue
    
    # Find CSV files in the isolated run directory
    run_dir = Path(output['output_directory'])
    csv_files = list(run_dir.glob("*.csv"))
    
    if csv_files:
        # Find the main dataset file (look for chembl in name)
        dataset_file = None
        for csv_file in csv_files:
            if 'chembl' in csv_file.name.lower():
                dataset_file = csv_file
                break
        
        # Fallback to first CSV if no chembl file found
        if dataset_file is None and csv_files:
            dataset_file = csv_files[0]
        
        if dataset_file:
            try:
                df = pd.read_csv(dataset_file)
                datasets.append({
                    'run_id': output['run_id'],
                    'prompt': output['prompt'],
                    'dataframe': df,
                    'file_path': dataset_file
                })
                print(f"Run {output['run_id']+1}: Loaded {len(df)} rows from {dataset_file.name}")
            except Exception as e:
                print(f"Run {output['run_id']+1}: Failed to load {dataset_file.name} - {e}")

if len(datasets) >= 2:
    dfs = [d['dataframe'] for d in datasets]
    data_comparison = comparator.compare_dataframes(dfs)
    
    print("\nDataset Comparison Results:")
    print("="*60)
    for metric, value in data_comparison.items():
        if isinstance(value, (int, float)):
            print(f"  {metric:.<40} {value:.3f}")
        else:
            print(f"  {metric:.<40} {value}")
    
    # Show detailed comparison
    print(f"\nDataset Details:")
    for ds in datasets:
        print(f"\nRun {ds['run_id']+1}:")
        print(f"  Shape: {ds['dataframe'].shape}")
        print(f"  Columns: {list(ds['dataframe'].columns)}")
        print(f"  File: {ds['file_path'].name}")
else:
    print("\nNot enough datasets to compare")
    print(f"Loaded {len(datasets)} dataset(s) from {len(outputs)} run(s)")





Not enough datasets to compare


In [23]:
# Calculate overall robustness score
if len(outputs) >= 2:
    comparison_results = {
        "text": text_comparison,
        "process": {
            "completion_rate": len(outputs) / len(chembl_variations),
            "tool_sequence_similarity": 0.90  # Placeholder
        }
    }
    
    if len(datasets) >= 2:
        comparison_results["data"] = data_comparison
    
    score = metrics_calc.calculate_robustness_score(comparison_results)
    rating = metrics_calc.get_rating(score)
    
    print(f"\n{'='*60}")
    print(f"ROBUSTNESS SCORE: {score:.3f} / 1.00")
    print(f"RATING: {rating}")
    print(f"{'='*60}")
else:
    print("\nNot enough successful runs to calculate robustness score")


2025-12-29 14:35:05,647 - metrics - INFO - Calculated robustness score: 0.305



ROBUSTNESS SCORE: 0.305 / 1.00
RATING: Concerning


In [ ]:
# Visualize and create detailed report (with MinIO/S3 support)
import matplotlib.pyplot as plt

# 1. Visualize file counts across runs
successful_outputs = [o for o in outputs if 'generated_files' in o]
if successful_outputs:
    runs = [o['run_id'] + 1 for o in successful_outputs]
    local_file_counts = [len(o['generated_files']) for o in successful_outputs]
    s3_file_counts = [len(o.get('s3_files', {})) for o in successful_outputs]
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Local files bar chart
    axes[0, 0].bar(runs, local_file_counts, color='steelblue', alpha=0.7)
    axes[0, 0].set_xlabel('Run Number', fontsize=12)
    axes[0, 0].set_ylabel('Number of Local Files', fontsize=12)
    axes[0, 0].set_title('Local Files per Run', fontsize=14, fontweight='bold')
    axes[0, 0].grid(axis='y', alpha=0.3)
    axes[0, 0].set_xticks(runs)
    
    # S3 files bar chart
    if s3_enabled:
        axes[0, 1].bar(runs, s3_file_counts, color='orange', alpha=0.7)
        axes[0, 1].set_xlabel('Run Number', fontsize=12)
        axes[0, 1].set_ylabel('Number of S3 Files', fontsize=12)
        axes[0, 1].set_title('S3/MinIO Files per Run', fontsize=14, fontweight='bold')
        axes[0, 1].grid(axis='y', alpha=0.3)
        axes[0, 1].set_xticks(runs)
    else:
        axes[0, 1].text(0.5, 0.5, 'S3/MinIO not enabled', ha='center', va='center')
        axes[0, 1].set_title('S3/MinIO Files per Run', fontsize=14, fontweight='bold')
    
    # Status pie chart
    statuses = []
    for o in outputs:
        if 'error' in o:
            statuses.append('Failed')
        elif o.get('status') == 'interrupted':
            statuses.append('Interrupted')
        else:
            statuses.append('Success')
    
    status_counts = {s: statuses.count(s) for s in set(statuses)}
    colors = {'Success': '#2ecc71', 'Failed': '#e74c3c', 'Interrupted': '#f39c12'}
    axes[1, 0].pie(status_counts.values(), labels=status_counts.keys(), autopct='%1.1f%%',
            colors=[colors.get(k, 'gray') for k in status_counts.keys()],
            startangle=90)
    axes[1, 0].set_title('Run Status Distribution', fontsize=14, fontweight='bold')
    
    # File size comparison (if files exist)
    if successful_outputs:
        file_sizes = []
        for o in successful_outputs:
            total_size = 0
            for local_path in o['generated_files'].values():
                if Path(local_path).exists():
                    total_size += Path(local_path).stat().st_size
            file_sizes.append(total_size / (1024 * 1024))  # Convert to MB
        
        axes[1, 1].bar(runs, file_sizes, color='green', alpha=0.7)
        axes[1, 1].set_xlabel('Run Number', fontsize=12)
        axes[1, 1].set_ylabel('Total Size (MB)', fontsize=12)
        axes[1, 1].set_title('File Size per Run', fontsize=14, fontweight='bold')
        axes[1, 1].grid(axis='y', alpha=0.3)
        axes[1, 1].set_xticks(runs)
    
    plt.tight_layout()
    viz_path = base_output_dir / 'run_comparison.png'
    plt.savefig(viz_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Visualization saved to: {viz_path}")

# 2. Generate detailed markdown report with S3 information
report_lines = [
    f"# Robustness Test Report",
    f"",
    f"**Test ID**: `{test_run_id}`",
    f"**Test Type**: `chembl_download`",
    f"**Date**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"**Total Runs**: {len(chembl_variations)}",
    f"**Successful**: {sum(1 for o in outputs if 'error' not in o and o.get('status') != 'interrupted')}",
    f"**Failed**: {sum(1 for o in outputs if 'error' in o)}",
    f"**Interrupted**: {sum(1 for o in outputs if o.get('status') == 'interrupted')}",
    f"",
    f"## Storage Configuration",
    f"",
    f"**S3/MinIO Enabled**: {'✓ Yes' if s3_enabled else '✗ No'}",
]

if s3_enabled:
    report_lines.extend([
        f"**S3 Bucket**: `{s3_config.bucket_name}`",
        f"**S3 Endpoint**: `{s3_config.endpoint_url}`",
    ])

report_lines.extend([
    f"",
    f"## Output Locations",
    f"",
    f"**Local Directory**: `{base_output_dir}`",
])

if s3_enabled:
    report_lines.append(f"**S3 Base Path**: `s3://{s3_config.bucket_name}/sessions/`")

report_lines.extend([
    f"",
    f"## Run Details",
    f""
])

for output in outputs:
    status_emoji = "✅" if 'error' not in output and output.get('status') != 'interrupted' else ("⚠️" if output.get('status') == 'interrupted' else "❌")
    report_lines.extend([
        f"### {status_emoji} Run {output['run_id'] + 1}",
        f"",
        f"**Prompt**:",
        f"```",
        output['prompt'],
        f"```",
        f"",
        f"**Session ID**: `{output['session_id']}`",
        f"",
        f"**Local Output Directory**: `{output['output_directory']}`",
    ])
    
    if output.get('s3_prefix'):
        report_lines.extend([
            f"",
            f"**S3 Prefix**: `{output['s3_prefix']}`",
        ])
    
    report_lines.extend([
        f"",
        f"**Local Files**: {len(output.get('generated_files', {}))}",
    ])
    
    if output.get('s3_files'):
        report_lines.append(f"**S3 Files**: {len(output['s3_files'])}")
    
    report_lines.append("")
    
    if 'error' in output:
        report_lines.extend([
            f"**Status**: Failed",
            f"",
            f"**Error**:",
            f"```",
            output['error'],
            f"```",
            f""
        ])
    elif output.get('status') == 'interrupted':
        report_lines.extend([
            f"**Status**: Interrupted by user",
            f""
        ])
    elif 'generated_files' in output:
        report_lines.extend([
            f"**Status**: Success",
            f"",
            f"**Generated Files**:",
        ])
        
        # Show local files
        report_lines.append(f"\n*Local Files:*")
        for file_key, file_path in output['generated_files'].items():
            file_size = Path(file_path).stat().st_size if Path(file_path).exists() else 0
            report_lines.append(f"- `{file_key}`: {Path(file_path).name} ({file_size:,} bytes)")
        
        # Show S3 files if available
        if output.get('s3_files'):
            report_lines.append(f"\n*S3/MinIO Files:*")
            for file_key, s3_url in output['s3_files'].items():
                report_lines.append(f"- `{file_key}`: {s3_url}")
        
        report_lines.append("")
        
        if 'result' in output:
            report_lines.extend([
                f"**Agent Response**:",
                f"```",
                output['result'][:500] + ("..." if len(output['result']) > 500 else ""),
                f"```",
                f""
            ])
    
    report_lines.append("---")
    report_lines.append("")

# Add comparison results if available
if len(datasets) >= 2:
    report_lines.extend([
        f"## Dataset Comparison",
        f"",
        f"Comparison metrics for {len(datasets)} datasets:",
        f""
    ])
    
    for metric, value in data_comparison.items():
        if isinstance(value, (int, float)):
            report_lines.append(f"- **{metric}**: {value:.3f}")
        else:
            report_lines.append(f"- **{metric}**: {value}")
    
    report_lines.append("")
    report_lines.append("### Dataset Details")
    report_lines.append("")
    
    for ds in datasets:
        report_lines.extend([
            f"**Run {ds['run_id']+1}**:",
            f"- Shape: {ds['dataframe'].shape[0]} rows × {ds['dataframe'].shape[1]} columns",
            f"- Columns: {', '.join(ds['dataframe'].columns[:10])}{'...' if len(ds['dataframe'].columns) > 10 else ''}",
            f"- File: `{ds['file_path'].name}`",
            f""
        ])

report_lines.extend([
    f"## File Inventory",
    f"",
    f"All generated files have been saved to isolated directories:",
    f""
])

for i in range(len(chembl_variations)):
    run_dir = base_output_dir / f"run_{i+1}"
    if run_dir.exists():
        files = list(run_dir.glob("*"))
        total_size = sum(f.stat().st_size for f in files if f.is_file())
        report_lines.append(f"- `run_{i+1}/`: {len(files)} file(s), {total_size / (1024*1024):.2f} MB")

report_content = "\n".join(report_lines)
report_path = base_output_dir / "DETAILED_REPORT.md"
report_path.write_text(report_content)

print(f"\n✓ Detailed report saved to: {report_path}")
print(f"\n{'='*60}")
print("Report Preview:")
print("="*60)
print(report_content[:1000])
print("\n[... See full report in file ...]")
print("="*60)

# 3. Create S3 access commands if S3 is enabled
if s3_enabled:
    commands_path = base_output_dir / "S3_ACCESS_COMMANDS.sh"
    commands = ["#!/bin/bash", "", "# Commands to access files in MinIO/S3", ""]
    
    for output in outputs:
        if 'error' not in output and output.get('s3_prefix'):
            session_id = output['session_id']
            commands.extend([
                f"# Run {output['run_id'] + 1} (Session: {session_id})",
                f"# List files:",
                f"mc ls myminio/sessions/{session_id}/",
                f"",
                f"# Download all files:",
                f"mc cp --recursive myminio/sessions/{session_id}/ ./run_{output['run_id'] + 1}_from_s3/",
                f"",
            ])
    
    commands_path.write_text("\n".join(commands))
    print(f"\n✓ S3 access commands saved to: {commands_path}")



## 5. Generate Full Report


In [ ]:
if len(outputs) >= 2:
    report = metrics_calc.generate_report({
        "score": score,
        "comparisons": comparison_results,
        "outliers": []
    })
    
    print(report)


In [ ]:
# Save report
report_dir = Path("../tests/robustness/reports")
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "notebook_test_report.md"

if len(outputs) >= 2:
    report_path.write_text(report)
    print(f"Report saved to: {report_path}")


## Summary

This notebook demonstrated the **Cs_copilot Robustness Testing Framework**:

### What You Can Do:

1. **Explore Prompt Variations** - View 10 different ways to request the same task
2. **Validate Semantic Similarity** - Ensure variations maintain intent (≥0.70 cosine similarity)
3. **Work with YAML** - Directly inspect and validate prompt templates
4. **Test Single Steps** - Check robustness of individual pipeline components
5. **Compare Outputs** - Automated comparison of datasets, text, images, and models
6. **Calculate Scores** - Weighted robustness score (0-1) with quality ratings
7. **Generate Reports** - Detailed markdown reports with recommendations
8. **Visualize Similarity** - Heatmaps showing semantic relationships between prompts

### Key Files:

- **`tests/robustness/prompt_variations.py`** - Prompt generation and validation
- **`tests/robustness/comparators.py`** - Multi-modal output comparison
- **`tests/robustness/metrics.py`** - Robustness scoring and reporting
- **`tests/robustness/fixtures/prompt_templates.yaml`** - 100+ prompt variations
- **`tests/robustness/test_pipeline_robustness.py`** - Pytest test suite

### Robustness Score Interpretation:

| Score | Rating | Meaning |
|-------|--------|---------|
| ≥0.90 | ✅ Excellent | Very robust to prompt variations |
| ≥0.80 | ✅ Good | Minor inconsistencies, acceptable for production |
| ≥0.70 | ⚠️ Acceptable | Some room for improvement, monitor closely |
| <0.70 | ❌ Concerning | Significant inconsistencies detected |

### Next Steps:

- Run with **n=10** variations for statistical significance
- Test across **different models** (GPT-4, Claude, DeepSeek)
- Integrate into **CI/CD** for regression testing
- Add **custom prompt categories** to the YAML file
- Create **automated dashboards** for tracking robustness over time

### Resources:

- [Framework README](../tests/robustness/README.md)
- [Implementation Summary](../tests/robustness/IMPLEMENTATION_SUMMARY.md)
- [Autoencoder Tests](../tests/robustness/AUTOENCODER_TESTS.md)


# Cs_copilot Robustness Testing

This notebook demonstrates how to use the robustness testing framework to assess how consistent the Cs_copilot pipeline is when given semantically equivalent but syntactically different prompts.

## What is Robustness Testing?

Robustness testing checks whether the system produces consistent outputs when the same task is requested in different ways. For example:
- "Fetch kinase 2 inhibitor data" vs "Download CHEMBL data for kinase 2 inhibitors"
- "Build GTM" vs "Construct the manifold" vs "Create a GTM model"

The framework compares outputs across multiple runs and calculates a robustness score based on:
- **Data consistency** (40%): Are datasets identical?
- **Semantic similarity** (30%): Are text responses semantically equivalent?
- **Process consistency** (20%): Do runs complete successfully?
- **Visual similarity** (10%): Are visualizations consistent?


## Setup


In [1]:
%load_ext autoreload
%autoreload 2
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
import sys
import logging
from pathlib import Path
import pandas as pd

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


In [3]:
# Import Cs_copilot
from cs_copilot.agents import get_cs_copilot_agent_team
from cs_copilot.storage import S3


2025-12-27 13:36:36,612 - rdkit - INFO - Enabling RDKit 2025.09.1 jupyter extensions


SESSION_ID is not set
SESSION_ID: 20251227-123636-7d986d


2025-12-27 13:36:39,242 - cs_copilot.agents.registry - INFO - Registered factory for agent type: autoencoder
2025-12-27 13:36:39,243 - cs_copilot.agents.registry - INFO - Registered factory for agent type: autoencoder_gtm_sampling
2025-12-27 13:36:39,244 - cs_copilot.agents.registry - INFO - Registered factory for agent type: chembl_downloader
2025-12-27 13:36:39,245 - cs_copilot.agents.registry - INFO - Registered factory for agent type: gtm_activity_analysis
2025-12-27 13:36:39,246 - cs_copilot.agents.registry - INFO - Registered factory for agent type: gtm_chemotype_analysis
2025-12-27 13:36:39,247 - cs_copilot.agents.registry - INFO - Registered factory for agent type: gtm_density_analysis
2025-12-27 13:36:39,247 - cs_copilot.agents.registry - INFO - Registered factory for agent type: gtm_loading
2025-12-27 13:36:39,249 - cs_copilot.agents.registry - INFO - Registered factory for agent type: gtm_optimization


In [ ]:
# Import robustness testing framework
sys.path.insert(0, str(Path.cwd().parent / 'tests' / 'robustness'))
from prompt_variations import PromptVariationGenerator
from comparators import OutputComparator
from metrics import RobustnessMetrics


In [ ]:
# Setup API key
from getpass import getpass
deepseek_api_key = getpass("DeepSeek API key: ")


In [ ]:
from agno.models.deepseek import DeepSeek
model = DeepSeek(id="deepseek-chat", api_key=deepseek_api_key)


## 1. Explore Available Prompt Variations


In [ ]:
# Initialize the prompt variation generator
generator = PromptVariationGenerator()

# List all available prompt categories
available_prompts = generator.list_available_prompts()
print("Available prompt categories:")
for prompt_key in available_prompts:
    print(f"  - {prompt_key}")


In [ ]:
# Get variations for a specific category
prompt_key = "chembl_download"  # Change this to try different categories
variations = generator.get_variations(prompt_key, n=5)

print(f"\n{len(variations)} variations for '{prompt_key}':\n")
for i, variation in enumerate(variations, 1):
    print(f"{i}. {variation}")
    print()


## 2. Validate Semantic Similarity of Variations

The framework uses sentence transformers to ensure variations maintain semantic similarity to the base prompt.


In [ ]:
# Check semantic similarity
base = variations[0]
print(f"Base prompt: {base}\n")

for i, var in enumerate(variations[1:], 2):
    is_valid = generator.validate_variation(base, var, min_similarity=0.70)
    print(f"Variation {i}: {'✅ Valid' if is_valid else '❌ Invalid'}")
    print(f"  {var}\n")


## 3. Example: Test Single Step Robustness

Let's test robustness of a single step (ChEMBL data download) with multiple prompt variations.


In [ ]:
# Get variations for ChEMBL download
chembl_variations = generator.get_variations("chembl_download", n=3)

# Initialize comparator and metrics
comparator = OutputComparator()
metrics_calc = RobustnessMetrics()

# Store outputs from each run
outputs = []

print("Testing ChEMBL download robustness with 3 variations...\n")


In [ ]:
# Run each variation
for i, prompt in enumerate(chembl_variations):
    print(f"\n{'='*60}")
    print(f"Run {i+1}/3")
    print(f"Prompt: {prompt}")
    print(f"{'='*60}\n")
    
    # Create fresh agent for each run
    agent = get_cs_copilot_agent_team(
        model=model,
        debug_mode=False,
        stream_intermediate_steps=False,
        show_members_responses=False
    )
    
    try:
        result = agent.run(prompt, stream=False)
        session_state = agent.get_session_state()
        
        # Collect output
        output = {
            "run_id": i,
            "prompt": prompt,
            "result": result.content,
            "session_state": session_state,
        }
        outputs.append(output)
        
        print(f"✅ Run {i+1} completed successfully")
        
    except Exception as e:
        print(f"❌ Run {i+1} failed: {e}")
        continue


In [ ]:
# Compare text outputs
if len(outputs) >= 2:
    texts = [out["result"] for out in outputs]
    text_comparison = comparator.compare_text_outputs(texts)
    
    print("\nText Comparison Results:")
    print("="*60)
    for metric, value in text_comparison.items():
        if isinstance(value, (int, float)):
            print(f"  {metric:.<40} {value:.3f}")
        else:
            print(f"  {metric:.<40} {value}")
else:
    print("Not enough successful runs to compare")


In [ ]:
# Compare datasets if available
datasets = []
for out in outputs:
    try:
        dataset_path = out["session_state"].get("data_file_paths", {}).get("dataset_path")
        if dataset_path and Path(dataset_path).exists():
            df = pd.read_csv(dataset_path)
            datasets.append(df)
            print(f"Run {out['run_id']+1}: Loaded dataset with {len(df)} rows")
    except Exception as e:
        print(f"Run {out['run_id']+1}: Could not load dataset - {e}")

if len(datasets) >= 2:
    data_comparison = comparator.compare_dataframes(datasets)
    
    print("\nDataset Comparison Results:")
    print("="*60)
    for metric, value in data_comparison.items():
        if isinstance(value, (int, float)):
            print(f"  {metric:.<40} {value:.3f}")
        else:
            print(f"  {metric:.<40} {value}")
else:
    print("\nNot enough datasets to compare")


In [ ]:
# Calculate overall robustness score
if len(outputs) >= 2:
    comparison_results = {
        "text": text_comparison,
        "process": {
            "completion_rate": len(outputs) / len(chembl_variations),
            "tool_sequence_similarity": 0.90  # Placeholder
        }
    }
    
    if len(datasets) >= 2:
        comparison_results["data"] = data_comparison
    
    score = metrics_calc.calculate_robustness_score(comparison_results)
    rating = metrics_calc.get_rating(score)
    
    print(f"\n{'='*60}")
    print(f"ROBUSTNESS SCORE: {score:.3f} / 1.00")
    print(f"RATING: {rating}")
    print(f"{'='*60}")
else:
    print("\nNot enough successful runs to calculate robustness score")


## 4. Generate Full Report


In [ ]:
if len(outputs) >= 2:
    report = metrics_calc.generate_report({
        "score": score,
        "comparisons": comparison_results,
        "outliers": []
    })
    
    print(report)


In [ ]:
# Save report
report_dir = Path("../tests/robustness/reports")
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "notebook_test_report.md"

if len(outputs) >= 2:
    report_path.write_text(report)
    print(f"Report saved to: {report_path}")


## 5. Advanced: Test Full Pipeline Robustness

This tests the entire workflow with multiple variations.


In [ ]:
# Get full pipeline variations
full_pipeline_variations = generator.get_variations("full_pipeline", n=3)

print("Full Pipeline Variations:")
for i, var in enumerate(full_pipeline_variations, 1):
    print(f"\n{i}. {var[:200]}...")


In [ ]:
# Run full pipeline with variations
full_outputs = []

for i, prompt in enumerate(full_pipeline_variations):
    print(f"\n{'='*60}")
    print(f"Full Pipeline Run {i+1}/{len(full_pipeline_variations)}")
    print(f"{'='*60}\n")
    
    # Create fresh agent
    agent = get_cs_copilot_agent_team(
        model=model,
        debug_mode=False,
        stream_intermediate_steps=False,
        show_members_responses=False
    )
    
    try:
        result = agent.run(prompt, stream=False)
        session_state = agent.get_session_state()
        
        output = {
            "run_id": i,
            "prompt": prompt,
            "result": result.content,
            "session_state": session_state,
        }
        full_outputs.append(output)
        
        print(f"✅ Run {i+1} completed successfully")
        print(f"   Files generated: {list(session_state.keys())}")
        
    except Exception as e:
        print(f"❌ Run {i+1} failed: {e}")
        continue

print(f"\nCompleted {len(full_outputs)}/{len(full_pipeline_variations)} runs")


In [ ]:
# Comprehensive comparison of full pipeline outputs
if len(full_outputs) >= 2:
    full_comparison = {}
    
    # Compare text
    texts = [out["result"] for out in full_outputs]
    full_comparison["text"] = comparator.compare_text_outputs(texts)
    
    # Compare datasets
    datasets = []
    for out in full_outputs:
        try:
            dataset_path = out["session_state"].get("data_file_paths", {}).get("dataset_path")
            if dataset_path and Path(dataset_path).exists():
                datasets.append(pd.read_csv(dataset_path))
        except:
            pass
    
    if len(datasets) >= 2:
        full_comparison["data"] = comparator.compare_dataframes(datasets)
    
    # Compare visualizations
    gtm_plots = []
    for out in full_outputs:
        try:
            plot_path = out["session_state"].get("gtm_file_paths", {}).get("gtm_plot_path")
            if plot_path and Path(plot_path).exists():
                gtm_plots.append(Path(plot_path))
        except:
            pass
    
    if len(gtm_plots) >= 2:
        full_comparison["visual"] = comparator.compare_images(gtm_plots)
    
    # Process metrics
    full_comparison["process"] = {
        "completion_rate": len(full_outputs) / len(full_pipeline_variations),
        "tool_sequence_similarity": 0.90
    }
    
    # Calculate score
    full_score = metrics_calc.calculate_robustness_score(full_comparison)
    full_rating = metrics_calc.get_rating(full_score)
    
    print(f"\n{'='*60}")
    print(f"FULL PIPELINE ROBUSTNESS SCORE: {full_score:.3f} / 1.00")
    print(f"RATING: {full_rating}")
    print(f"{'='*60}")
    
    # Generate report
    full_report = metrics_calc.generate_report({
        "score": full_score,
        "comparisons": full_comparison,
        "outliers": []
    })
    
    print("\n" + full_report)
else:
    print("Not enough successful runs to perform full comparison")


## 6. Custom Weights and Thresholds

You can customize the importance of different metrics and the thresholds for pass/fail.


In [ ]:
# Create custom metrics calculator
custom_metrics = RobustnessMetrics(
    weights={
        "data_similarity": 0.5,      # Emphasize data consistency
        "semantic_similarity": 0.3,
        "process_consistency": 0.15,
        "visual_similarity": 0.05,   # De-emphasize visual similarity
    },
    thresholds={
        "excellent": 0.95,
        "good": 0.85,
        "acceptable": 0.75,
    }
)

print("Custom Configuration:")
print("\nWeights:")
for key, value in custom_metrics.weights.items():
    print(f"  {key:.<40} {value:.2f}")

print("\nThresholds:")
for key, value in custom_metrics.thresholds.items():
    print(f"  {key:.<40} {value:.2f}")


In [ ]:
# Recalculate score with custom weights
if len(full_outputs) >= 2:
    custom_score = custom_metrics.calculate_robustness_score(full_comparison)
    custom_rating = custom_metrics.get_rating(custom_score)
    
    print(f"\nStandard Score: {full_score:.3f} ({full_rating})")
    print(f"Custom Score:   {custom_score:.3f} ({custom_rating})")
